# geo1261 – Mission plan for the Green Fab Lab

**Unit:** geo1260 Drone mapping · **Slides:** `geo1260_Drone_Imagery_WebODM` (part *Plan*) · **Data:** `data/geo1261/gfl_flight_area.gpkg` · **Homework**

Before a mapping flight the planning app computes how high to fly, how far apart the flight lines are, where the camera takes a photo, how fast the drone may fly and how long the flight takes. In this notebook you compute all of it yourself for the Mavic 3M over the Green Fab Lab, draw the flight lines on a map and export the area for the flight app (DJI Pilot 2). In the field you compare your numbers with the app.

You will learn to
1. derive the ground sampling distance (GSD) from the camera specs,
2. turn height and overlap into line spacing, photo spacing, speed and flight time,
3. generate parallel flight lines over a polygon with Shapely,
4. explain why the multispectral camera decides the plan when all cameras record.

About 2–3 h. The stretch tasks are marked.

## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely import affinity
from shapely.geometry import LineString, MultiLineString, Point, box

DATA = Path("data") / "geo1261"
area_gdf = gpd.read_file(DATA / "gfl_flight_area.gpkg", layer="flight_area")
bld_gdf = gpd.read_file(DATA / "gfl_flight_area.gpkg", layer="building")
area = area_gdf.geometry.iloc[0]           # shapely Polygon in EPSG:25832 (metres)
print(area_gdf.crs, f"{area.area:.0f} m²")

ax = area_gdf.plot(color="pink", edgecolor="crimson")
bld_gdf.plot(ax=ax, color="grey")
ax.set_title("Green Fab Lab: flight area and building");

## 1. Camera: pixel size on the ground

DJI Mavic 3M specs (ag.dji.com/mavic-3-m/specs):

| | RGB | multispectral (each band) |
|---|---|---|
| image size | 5280 × 3956 px | 2592 × 1944 px |
| field of view | 84° **diagonal** | 61.2° × 48.1° (horizontal × vertical) |
| shortest photo interval | 0.7 s (JPEG) | 2 s (TIFF) |

A camera with field of view $\alpha$ at height $H$ sees a ground width of $2H\tan(\alpha/2)$. Divided by the number of pixels across that width you get the **ground sampling distance** (GSD).

**Exercise 1:** complete `CAMERAS` with `gsd_per_m` (GSD in metres per metre of height) for both cameras, and write `gsd(cam, height)`. Mind which field of view goes with which pixel count.

In [ ]:
CAMERAS = {
    "RGB": dict(px=(5280, 3956), interval=0.7),
    "MS": dict(px=(2592, 1944), interval=2.0),
}
# YOUR CODE HERE

for h in (25, 40, 60, 120):
    print(f"{h:4d} m: RGB {gsd('RGB', h) * 100:.2f} cm, MS {gsd('MS', h) * 100:.2f} cm")

**Check:** at 100 m the RGB pixel should be about 2.7 cm. Why is the multispectral pixel larger?

✍️ *Your answer:*

## 2. From height and overlap to spacing and speed

We assume the long image side lies **across** the flight direction (as in DJI's mapping missions).

* footprint across = pixels across × GSD, footprint along = pixels along × GSD
* line spacing = footprint across × (1 − side overlap)
* photo spacing ("base") = footprint along × (1 − front overlap)
* speed = photo spacing / shortest photo interval, at most `V_MAX`

**Exercise 2:** write `plan_numbers(cam, height, front, side)` that returns a dict with `gsd`, `across`, `along`, `spacing`, `base` and `v`.

In [ ]:
V_MAX = 15.0    # m/s

# YOUR CODE HERE

H = 25
pd.DataFrame({cam: plan_numbers(cam, H) for cam in CAMERAS}).round(3)

**Questions:** Both cameras record at the same time. Which camera sets the line spacing, which the photo spacing and which the speed? What do you use for the flight?

✍️ *Your answer:*

## 3. Flight lines over the area

Idea: rotate the area so that the flight direction becomes the x axis, draw horizontal lines `spacing` apart, cut them with the rotated area, rotate them back.

The flight direction should follow the **long side** of the area (fewer turns). `heading_of(area)` gives that direction in degrees from east (given).

**Exercise 3:** write `flight_lines(area, spacing, heading)` that returns a list of `LineString`s, every second line reversed (the drone flies back and forth). Centre the lines across the area. Hint: `affinity.rotate(geom, angle, origin=point)`, `LineString.intersection(polygon)`; if the result is a `MultiLineString`, keep the longest part.

In [ ]:
def heading_of(area):
    # Direction of the longest edge of the minimum rotated rectangle, degrees from east.
    rr = list(area.minimum_rotated_rectangle.exterior.coords)
    edges = [(rr[i], rr[i + 1]) for i in range(4)]
    (x0, y0), (x1, y1) = max(edges, key=lambda e: np.hypot(e[1][0] - e[0][0], e[1][1] - e[0][1]))
    return np.degrees(np.arctan2(y1 - y0, x1 - x0))


# YOUR CODE HERE

heading = heading_of(area)
both = [plan_numbers(cam, H) for cam in CAMERAS]
spacing = min(p["spacing"] for p in both)
lines = flight_lines(area, spacing, heading)
print(f"heading {heading:.0f}°, {len(lines)} lines, {spacing:.1f} m apart")

## 4. Photos, path and flight time

**Exercise 4:** compute
* `photos`: photo positions every `base` metres along each line, starting at the start of the line (`LineString.interpolate`),
* `length`: path length = all lines + the short connections between the end of one line and the start of the next,
* `time`: flight time in minutes = `length / v` plus `TURN_S` seconds for each turn.

Use the minimum `base` and `v` of both cameras.

In [ ]:
TURN_S = 10.0   # s per turn (assumption)
base = min(p["base"] for p in both)
v = min(p["v"] for p in both)

# YOUR CODE HERE

print(f"{len(photos)} photo positions, path {length:.0f} m, {v:.1f} m/s, about {time:.1f} min")

## 5. Map and export

The map shows the area, the flight lines, the photo positions and the footprints of one RGB and one multispectral photo. Then we save the plan as a GeoPackage (for QGIS) and the **area** as KML for DJI Pilot 2 (the app plans its own lines inside it).

In [ ]:
def footprint(p, cam, height, heading):
    n = plan_numbers(cam, height)
    r = box(p.x - n["along"] / 2, p.y - n["across"] / 2, p.x + n["along"] / 2, p.y + n["across"] / 2)
    return affinity.rotate(r, heading, origin=p)


ax = area_gdf.plot(color="pink", edgecolor="crimson", figsize=(7, 7))
bld_gdf.plot(ax=ax, color="grey")
gpd.GeoSeries(lines, crs=25832).plot(ax=ax, color="navy")
gpd.GeoSeries(photos, crs=25832).plot(ax=ax, color="navy", markersize=6)
p = photos[len(photos) // 2]
gpd.GeoSeries([footprint(p, "RGB", H, heading)]).boundary.plot(ax=ax, color="grey")
gpd.GeoSeries([footprint(p, "MS", H, heading)]).boundary.plot(ax=ax, color="tab:blue", linestyle="--")
ax.set_title(f"Mission plan, {H} m, all cameras");

out = Path("geo1261_plan.gpkg")
gpd.GeoDataFrame({"line": range(1, len(lines) + 1)}, geometry=lines, crs=25832).to_file(out, layer="flight_lines")
gpd.GeoDataFrame({"photo": range(1, len(photos) + 1)}, geometry=photos, crs=25832).to_file(out, layer="photos")
area_gdf.to_crs(4326).to_file("gfl_flight_area.kml", driver="KML")   # for DJI Pilot 2 (WGS 84)
print("written", out, "and gfl_flight_area.kml")

## 6. Compare heights

**Exercise 5:** wrap §2–§4 into `mission(height, cams)` and make a table for 25, 30, 40 and 60 m: pixel size RGB and MS, number of lines, photo positions and flight time, once for all cameras and once for RGB only.

In [ ]:
# YOUR CODE HERE
table.round(2)

**Questions**
1. Which height would you choose for the Green Fab Lab, and why? Think of pixel size, the size of the area (how much of a photo lies outside it?), the trees and the battery.
2. At 25 m a photo covers more than a third of the area. What does that mean for the photos at the edge of the area, and for the GCPs near the edge?
3. Stretch: change the overlap to 83 % / 75 % (vegetation, ODM). How much longer is the flight?
4. In the field: write down DJI Pilot 2's numbers for the same height and overlap (line spacing, number of photos, time). Where do they differ from yours, and why?

✍️ *Your answer:*